# Notebook 46 — Cross-Distribution Multi-Probe Ensemble (v2)

## The Gate

Does the +6.7pp ensemble lift from nb45 (in-distribution) generalize to OOD datasets neither probe was trained on?

**Decides simultaneously**:
- Paper-4 (ProbePack ensemble) publishable?
- Paper-5 (Probe-Gated Memory) foundation sound?
- ProbeBench v0.0.2 ensemble axis legitimate?

## Datasets (450 prompts total)

| Dataset | n | OOD axis | Predicted FG | Predicted RG |
|---|---|---|---|---|
| TruthfulQA gen | 150 | factual+adversarial | 0.65-0.78 | 0.50-0.58 (noise) |
| StrategyQA | 150 | multi-step+factual | 0.55-0.65 | 0.55-0.62 |
| TriviaQA rc.nocontext | 150 | pure factual | 0.72-0.80 | 0.50-0.55 (noise) |

## Per-dataset verdict

`lift = best_ensemble_AUROC − max(FG_AUROC, RG_AUROC)` with paired bootstrap CI.

| Verdict | Criterion |
|---|---|
| 🟢 survives | lift ≥ +3pp AND CI excludes 0 |
| ⚪ null | \|lift\| < 3pp OR CI crosses 0 |
| 🔴 hurts | lift ≤ −3pp AND CI excludes 0 |

## Aggregate verdict (5 states)

| State | Criterion | Action |
|---|---|---|
| 🟢 STRONG | 3/3 survive, mean lift > +3pp, CI excludes 0 | Paper-4 + paper-5 ship. ProbePack vendable. |
| 🟢 STANDARD | 2/3 survive, mean lift ≥ +2pp | Paper-4 with declared scope. ProbePack honest. |
| 🟡 MIXED | 1/3 survive, no hurts | Paper-4 weak. ProbePack needs guardrails. |
| ⚪ NULL | 0/3 survive, mean ≈ 0 | Walk back ensemble claim. Single probes only. |
| 🔴 HURTS | ≥1 hurts | Pull ensemble axis from ProbeBench. Walk back public. |

**Compute**: ~6h on RTX 6000 (~R$27).

**Drive**: `/content/drive/MyDrive/openinterp_runs/46_cross_distribution_ensemble/`

---

Diagnostic checkpoints (catch silent failures early):
- Phase 2: activation-norm sanity vs FG/RG training distribution
- Phase 4 mid-run: has_think rate per source — abort if <30% on any source after 30 prompts
- Phase 5: judge UNVERIFIABLE rate per source
- Phase 6: RG calibration cross-check vs nb32 (StrategyQA AUROC ≈ 0.605 ± 0.10 expected)
- Phase 6: fusion-method consistency (does the same method win across datasets?)


## Phase 1 — Setup + Drive mount


In [ ]:
from pathlib import Path
import os, json, time
import torch, numpy as np

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
OUT = DRIVE / 'openinterp_runs' / '46_cross_distribution_ensemble'
OUT.mkdir(parents=True, exist_ok=True)
print(f'OUT: {OUT}')
print(f'Existing files: {sorted(p.name for p in OUT.iterdir())}')


In [ ]:
!pip install -q -U torchao
!pip install -q -U transformers accelerate datasets
!pip install -q -U huggingface_hub
!pip install -q openai scikit-learn joblib matplotlib
print('✓ deps')


## Phase 2 — Qwen3.6-27B + FG + RG probes

Includes activation-norm sanity check: if OOD activations are 2x larger or smaller
than the FG/RG training distribution, the StandardScaler will misnormalize and
probe scores become uncalibrated. Catches silent OOD-shift failures.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, hf_hub_download, HfApi, create_repo
import getpass, joblib

CFG = {
    'model_id':              'Qwen/Qwen3.6-27B',
    'capture_layer_fg':      31,
    'capture_layer_rg':      55,
    'n_per_dataset':         150,
    'temperature':           0.7,
    'max_new_tokens':        2048,
    'random_seed':           46,
    'fg_probe_repo':         'caiovicentino1/FabricationGuard-linearprobe-qwen36-27b',
    'rg_probe_repo':         'caiovicentino1/ReasoningGuard-linearprobe-qwen36-27b',
    'judge_model':           'anthropic/claude-haiku-4.5',
    'output_repo':           'caiovicentino1/openinterp-46-cross-distribution-ensemble',
    'fusion_alpha':          0.5,
    'bootstrap_n':           1000,
    'lift_threshold_pp':     0.03,
    'mid_run_check_after':   30,
    'min_has_think_rate':    0.30,
    'rg_strategyqa_ref':     0.605,
}
THINK_CLOSE_ID = 248069
torch.manual_seed(CFG['random_seed']); np.random.seed(CFG['random_seed'])

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
login(HF_TOKEN, add_to_git_credential=False)
OPENROUTER_KEY = os.environ.get('OPENROUTER_API_KEY') or getpass.getpass('OpenRouter API key: ')
os.environ['OPENROUTER_API_KEY'] = OPENROUTER_KEY

device = 'cuda'
tok = AutoTokenizer.from_pretrained(CFG['model_id'])
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'], torch_dtype=torch.bfloat16, device_map='auto',
)
model.eval()
print(f'✓ Base loaded — {torch.cuda.get_device_name(0)}')


In [ ]:
fg_path = hf_hub_download(repo_id=CFG['fg_probe_repo'], filename='probe.joblib', repo_type='dataset')
rg_path = hf_hub_download(repo_id=CFG['rg_probe_repo'], filename='probe.joblib', repo_type='dataset')
fg_artifact = joblib.load(fg_path)
rg_artifact = joblib.load(rg_path)
fg_clf = fg_artifact['probe']; fg_scaler = fg_artifact['scaler']
rg_clf = rg_artifact['probe']; rg_scaler = rg_artifact['scaler']
if not hasattr(fg_clf, 'multi_class'): fg_clf.multi_class = 'auto'
if not hasattr(rg_clf, 'multi_class'): rg_clf.multi_class = 'auto'

# Training-distribution stats (from scaler) — used for OOD-shift sanity.
fg_train_mean_norm = float(np.linalg.norm(fg_scaler.mean_))
fg_train_scale_mean = float(fg_scaler.scale_.mean())
rg_train_mean_norm = float(np.linalg.norm(rg_scaler.mean_))
rg_train_scale_mean = float(rg_scaler.scale_.mean())
print(f'FG scaler: ||mean||={fg_train_mean_norm:.2f}, mean(scale)={fg_train_scale_mean:.4f}')
print(f'RG scaler: ||mean||={rg_train_mean_norm:.2f}, mean(scale)={rg_train_scale_mean:.4f}')

def fg_score(act):
    x = act.float().cpu().numpy().reshape(1, -1)
    return float(fg_clf.predict_proba(fg_scaler.transform(x))[0, 1])
def rg_score(act):
    x = act.float().cpu().numpy().reshape(1, -1)
    return float(rg_clf.predict_proba(rg_scaler.transform(x))[0, 1])
print('✓ FG + RG probes loaded')


## Phase 3 — Forward hooks + 3 OOD datasets


In [ ]:
captured = {}
_pos = {'pos': None}

def make_hook(layer_idx):
    def hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        pos = _pos['pos']
        if pos is None or pos >= h.shape[1]: return
        captured[f'L{layer_idx}'] = h[0, pos, :].detach().cpu().to(torch.float16).clone()
    return hook

hook_handles = []
for L in [CFG['capture_layer_fg'], CFG['capture_layer_rg']]:
    h = model.model.layers[L].register_forward_hook(make_hook(L))
    hook_handles.append(h)
print(f'✓ Hooks at L{CFG["capture_layer_fg"]}, L{CFG["capture_layer_rg"]}')


In [ ]:
from datasets import load_dataset
rng = np.random.default_rng(CFG['random_seed'])
N = CFG['n_per_dataset']

# TruthfulQA generation: factual + adversarial, has best_answer + correct/incorrect refs.
tqa = load_dataset('truthfulqa/truthful_qa', 'generation', split='validation')
tqa_idx = rng.choice(len(tqa), size=min(N, len(tqa)), replace=False)
tqa_pool = []
for i in tqa_idx:
    ex = tqa[int(i)]
    tqa_pool.append({
        'id': f'tqa_{i}', 'src': 'truthfulqa',
        'question': ex['question'],
        'gold': ex['best_answer'],
        'correct_refs': ex.get('correct_answers', []),
        'incorrect_refs': ex.get('incorrect_answers', []),
    })
print(f'TruthfulQA: {len(tqa_pool)}')

# StrategyQA: yes/no multi-step. Boolean answer.
try:
    sqa = load_dataset('ChilleD/StrategyQA', split='test')
except Exception:
    sqa = load_dataset('voidful/StrategyQA', split='train')
sqa_idx = rng.choice(len(sqa), size=min(N, len(sqa)), replace=False)
sqa_pool = []
for i in sqa_idx:
    ex = sqa[int(i)]
    ans = ex.get('answer')
    answer_bool = ans if isinstance(ans, bool) else (str(ans).strip().lower() in ('true', 'yes', '1'))
    sqa_pool.append({
        'id': f'stq_{i}', 'src': 'strategyqa',
        'question': ex['question'],
        'gold': 'YES' if answer_bool else 'NO',
    })
print(f'StrategyQA: {len(sqa_pool)} (base rate YES: {sum(p["gold"]=="YES" for p in sqa_pool)/len(sqa_pool):.2%})')

# TriviaQA rc.nocontext: factual, has answer.aliases for matching.
trv = load_dataset('mandarjoshi/trivia_qa', 'rc.nocontext', split='validation')
trv_idx = rng.choice(len(trv), size=min(N, len(trv)), replace=False)
trv_pool = []
for i in trv_idx:
    ex = trv[int(i)]
    aliases = list(ex['answer'].get('aliases', [])) + [ex['answer'].get('value', '')]
    aliases = [a for a in aliases if a]
    trv_pool.append({
        'id': f'trv_{i}', 'src': 'triviaqa',
        'question': ex['question'],
        'gold': ex['answer']['value'],
        'aliases': aliases,
    })
print(f'TriviaQA: {len(trv_pool)}')

holdout = tqa_pool + sqa_pool + trv_pool
rng.shuffle(holdout)
src_dist = {}
for p in holdout: src_dist[p['src']] = src_dist.get(p['src'], 0) + 1
print(f'\nTotal hold-out: {len(holdout)}')
print(f'Source distribution: {src_dist}')


## Phase 4 — Generate + dual-probe scoring

Resume-safe (every record appended to Drive). Mid-run diagnostic prints has_think rate per source
every 30 prompts so you can ctrl-C early if a dataset is failing to enter thinking mode.

Also tracks activation-norm OOD shift vs FG/RG training distribution — flags if shift > 25%.


In [ ]:
from tqdm.auto import tqdm
import gc
from collections import defaultdict

def find_end_think(token_ids):
    ids = token_ids.tolist() if hasattr(token_ids, 'tolist') else list(token_ids)
    for i in range(len(ids) - 1, -1, -1):
        if ids[i] == THINK_CLOSE_ID: return i
    return None

def generate_and_probe(prompt):
    messages = [{'role': 'user', 'content': prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    enc = tok(text, return_tensors='pt')
    ids = enc['input_ids'].to(device)
    amask = enc.get('attention_mask', torch.ones_like(ids)).to(device)
    n_in = ids.shape[1]
    with torch.no_grad():
        gen = model.generate(
            ids, attention_mask=amask,
            max_new_tokens=CFG['max_new_tokens'],
            do_sample=True, temperature=CFG['temperature'], top_p=0.95,
            pad_token_id=tok.eos_token_id,
        )
    full_ids = gen[0]
    output_ids = full_ids[n_in:]
    end_pos = find_end_think(full_ids)
    output_text = tok.decode(output_ids, skip_special_tokens=False)
    if '</think>' in output_text:
        cot = output_text.split('</think>', 1)[0].strip()
        answer = output_text.split('</think>', 1)[1].strip()
    else:
        cot, answer = output_text.strip(), ''
    if end_pos is None:
        return {'cot': cot, 'answer': answer, 'fg': None, 'rg': None,
                'has_think': False, 'act_norm_fg': None, 'act_norm_rg': None}
    captured.clear()
    _pos['pos'] = end_pos
    with torch.no_grad():
        _ = model(full_ids.unsqueeze(0).to(device))
    act_fg = captured.get(f'L{CFG["capture_layer_fg"]}')
    act_rg = captured.get(f'L{CFG["capture_layer_rg"]}')
    act_norm_fg = float(act_fg.float().norm().item()) if act_fg is not None else None
    act_norm_rg = float(act_rg.float().norm().item()) if act_rg is not None else None
    return {
        'cot': cot, 'answer': answer,
        'fg': fg_score(act_fg) if act_fg is not None else None,
        'rg': rg_score(act_rg) if act_rg is not None else None,
        'has_think': True, 'end_pos': end_pos,
        'act_norm_fg': act_norm_fg, 'act_norm_rg': act_norm_rg,
    }

results_path = OUT / 'generations_with_probes.jsonl'
done_keys = set()
if results_path.exists():
    with open(results_path) as f:
        for line in f:
            try: done_keys.add(json.loads(line)['id'])
            except: continue
    print(f'Resume: {len(done_keys)} done')

running_stats = defaultdict(lambda: {'total': 0, 'thinking': 0, 'norm_fg': [], 'norm_rg': []})
for i, p in enumerate(tqdm(holdout, desc='gen+probe')):
    if p['id'] in done_keys: continue
    try:
        torch.manual_seed(hash(p['id']) % (2**32))
        t0 = time.time()
        res = generate_and_probe(p['question'])
        gen_time = time.time() - t0
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        print(f'OOM on {p["id"]}'); continue
    record = {**p, **res, 'gen_time_s': gen_time}
    with open(results_path, 'a') as f:
        f.write(json.dumps(record) + '\n')
    done_keys.add(p['id'])
    running_stats[p['src']]['total'] += 1
    if res.get('has_think'):
        running_stats[p['src']]['thinking'] += 1
        if res.get('act_norm_fg'): running_stats[p['src']]['norm_fg'].append(res['act_norm_fg'])
        if res.get('act_norm_rg'): running_stats[p['src']]['norm_rg'].append(res['act_norm_rg'])
    if (i + 1) % CFG['mid_run_check_after'] == 0:
        print('\n--- mid-run check ---')
        for src, s in running_stats.items():
            tr = s['thinking'] / max(1, s['total'])
            mfg = float(np.mean(s['norm_fg'])) if s['norm_fg'] else 0
            mrg = float(np.mean(s['norm_rg'])) if s['norm_rg'] else 0
            warn = ''
            if s['total'] >= 10 and tr < CFG['min_has_think_rate']:
                warn = '⚠️  has_think rate critically low — consider abort'
            print(f'  {src}: n={s["total"]:3d} think={tr:.0%} ||L31||={mfg:.1f} ||L55||={mrg:.1f} {warn}')
        torch.cuda.empty_cache(); gc.collect()

print('\n✓ Phase 4 complete')


## Phase 4.5 — Diagnostic abort gate

Final has_think rate per source + activation-norm OOD shift vs FG/RG training stats.
If any source has has_think rate < 30%, the OOD test on that dataset is meaningless
(the probe fires on a non-thinking activation pattern it was never trained on).

If activation norm shifts > 25% from training, scaler is misnormalizing → probe scores
uncalibrated → AUROC measurement on that dataset is suspect.


In [ ]:
with open(results_path) as f:
    all_records = [json.loads(line) for line in f]

from collections import defaultdict
diag = defaultdict(lambda: {'total': 0, 'thinking': 0, 'norms_fg': [], 'norms_rg': []})
for r in all_records:
    s = r['src']
    diag[s]['total'] += 1
    if r.get('has_think'):
        diag[s]['thinking'] += 1
        if r.get('act_norm_fg'): diag[s]['norms_fg'].append(r['act_norm_fg'])
        if r.get('act_norm_rg'): diag[s]['norms_rg'].append(r['act_norm_rg'])

print(f'FG training ||mean||: {fg_train_mean_norm:.2f}')
print(f'RG training ||mean||: {rg_train_mean_norm:.2f}')
print()
print(f'{"source":<12} {"n":>4} {"think%":>8} {"||L31||":>10} {"shift_fg":>10} {"||L55||":>10} {"shift_rg":>10}')
print('-' * 72)
abort = []
for s, d in sorted(diag.items()):
    tr = d['thinking'] / max(1, d['total'])
    mfg = float(np.mean(d['norms_fg'])) if d['norms_fg'] else 0
    mrg = float(np.mean(d['norms_rg'])) if d['norms_rg'] else 0
    shift_fg = (mfg - fg_train_mean_norm) / fg_train_mean_norm if fg_train_mean_norm > 0 else 0
    shift_rg = (mrg - rg_train_mean_norm) / rg_train_mean_norm if rg_train_mean_norm > 0 else 0
    flags = []
    if tr < CFG['min_has_think_rate']: flags.append(f'low has_think ({tr:.0%})')
    if abs(shift_fg) > 0.25: flags.append(f'L31 norm shift ({shift_fg:+.0%})')
    if abs(shift_rg) > 0.25: flags.append(f'L55 norm shift ({shift_rg:+.0%})')
    print(f'{s:<12} {d["total"]:>4} {tr:>7.0%} {mfg:>10.2f} {shift_fg:>+9.0%} {mrg:>10.2f} {shift_rg:>+9.0%}')
    if flags: abort.append((s, flags))

if abort:
    print('\n⚠️  Diagnostic flags:')
    for s, flags in abort:
        print(f'  {s}: {", ".join(flags)}')
    print('\n  These results may be unreliable. Continue with caution.')
else:
    print('\n✓ All diagnostics within tolerances')


## Phase 5 — Ground truth (judge + gold-match per dataset type)

- TruthfulQA: judge with explicit correct/incorrect references.
- StrategyQA: gold-match YES/NO heuristic + judge fallback for ambiguous answers.
- TriviaQA: alias substring match + judge fallback.

Tracks UNVERIFIABLE rate per source (catches judge bias / weird answers).


In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_KEY,
)
import re

def normalize(s):
    return re.sub(r'[^a-z0-9 ]', ' ', s.lower()).strip()

def grade_strategyqa(answer, gold):
    a = answer.lower().strip()
    head = a[:300]
    has_yes = bool(re.search(r'\b(yes|true|correct|right)\b', head))
    has_no = bool(re.search(r'\b(no|false|incorrect|wrong)\b', head))
    if gold == 'YES':
        return has_yes and not has_no
    else:
        return has_no and not has_yes

def grade_triviaqa(answer, aliases):
    norm_ans = normalize(answer)
    for alias in aliases:
        if not alias: continue
        norm_alias = normalize(alias)
        if len(norm_alias) >= 2 and norm_alias in norm_ans:
            return True
    return False

JUDGE_TQA = '''Is this answer factually correct?
Reply ONE WORD: YES, NO, or UNVERIFIABLE.

Question: {q}
Correct reference answers: {correct}
Incorrect reference answers (common misconceptions): {incorrect}
Model answer: {a}'''

JUDGE_GENERIC = '''Is this answer factually correct?
Reply ONE WORD: YES, NO, or UNVERIFIABLE.

Question: {q}
Gold answer: {g}
Model answer: {a}'''

def judge_tqa(q, correct_refs, incorrect_refs, a):
    try:
        prompt = JUDGE_TQA.format(
            q=q,
            correct=' | '.join(correct_refs[:5]) if correct_refs else '(none)',
            incorrect=' | '.join(incorrect_refs[:5]) if incorrect_refs else '(none)',
            a=a[:1000],
        )
        r = client.chat.completions.create(
            model=CFG['judge_model'], max_tokens=80,
            messages=[{'role': 'user', 'content': prompt}],
        )
        t = r.choices[0].message.content.strip().upper()
        if t.startswith('YES'): return 'YES'
        if t.startswith('NO'): return 'NO'
        return 'UNVERIFIABLE'
    except Exception as e:
        return 'ERROR'

def judge_generic(q, g, a):
    try:
        r = client.chat.completions.create(
            model=CFG['judge_model'], max_tokens=80,
            messages=[{'role': 'user',
                       'content': JUDGE_GENERIC.format(q=q, g=g, a=a[:1000])}],
        )
        t = r.choices[0].message.content.strip().upper()
        if t.startswith('YES'): return 'YES'
        if t.startswith('NO'): return 'NO'
        return 'UNVERIFIABLE'
    except Exception as e:
        return 'ERROR'

scored_path = OUT / 'scored.jsonl'
scored_keys = set()
if scored_path.exists():
    with open(scored_path) as f:
        for line in f:
            try: scored_keys.add(json.loads(line)['id'])
            except: continue
    print(f'Resume scoring: {len(scored_keys)} done')

for r in tqdm(all_records, desc='score'):
    if r['id'] in scored_keys: continue
    if not r.get('has_think'): continue
    answer = (r.get('answer') or '').strip() or (r.get('cot') or '')[:500]
    src = r['src']
    if src == 'truthfulqa':
        label = judge_tqa(r['question'], r.get('correct_refs', []),
                          r.get('incorrect_refs', []), answer)
        is_correct = label == 'YES'
    elif src == 'strategyqa':
        is_correct = grade_strategyqa(answer, r['gold'])
        if not is_correct and 'yes' not in answer.lower()[:300] and 'no' not in answer.lower()[:300]:
            label = judge_generic(r['question'], r['gold'], answer)
            is_correct = label == 'YES'
        else:
            label = 'YES' if is_correct else 'NO'
    elif src == 'triviaqa':
        is_correct = grade_triviaqa(answer, r.get('aliases', [r['gold']]))
        if not is_correct:
            label = judge_generic(r['question'], r['gold'], answer)
            is_correct = label == 'YES'
        else:
            label = 'YES'
    else:
        label = 'UNVERIFIABLE'; is_correct = False
    out = {**r, 'is_correct': bool(is_correct), 'judge_label': label}
    with open(scored_path, 'a') as f:
        f.write(json.dumps(out) + '\n')
    scored_keys.add(r['id'])

print('\n✓ Phase 5 complete')


In [ ]:
with open(scored_path) as f:
    scored = [json.loads(line) for line in f]
import pandas as pd
sdf = pd.DataFrame(scored)
print('Judge label distribution per source:')
print(sdf.groupby(['src', 'judge_label']).size().unstack(fill_value=0))
print('\nIncorrect rate (target=positive class for AUROC):')
print(sdf.groupby('src')['is_correct'].agg(['mean', 'count']).round(3))

for src in sdf['src'].unique():
    sub = sdf[sdf['src'] == src]
    rate_unv = (sub['judge_label'] == 'UNVERIFIABLE').mean()
    if rate_unv > 0.20:
        print(f'⚠️  {src}: UNVERIFIABLE rate {rate_unv:.0%} — judge struggling, AUROC may be noisy')


## Phase 6 — Per-dataset AUROC + paired bootstrap lift CI

For each dataset, compute:
- AUROC of FG, RG, and 4 fusion methods
- Lift = best_ensemble − max(FG, RG)
- Paired bootstrap CI on lift (removes between-prompt variance — much tighter than independent CI)

Plus diagnostics:
- RG StrategyQA cross-check vs nb32 (0.605 ± 0.10)
- FG-RG correlation per dataset (low = orthogonal = ensemble has headroom)
- Fusion-method consistency (does best fusion method change across datasets?)


In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

df = pd.DataFrame(scored)
df = df[df['has_think'] & df['fg'].notnull() & df['rg'].notnull()].copy()
df['target'] = (~df['is_correct']).astype(int)  # 1 = incorrect (positive class)
df['fusion_weighted_avg'] = CFG['fusion_alpha'] * df['fg'] + (1 - CFG['fusion_alpha']) * df['rg']
df['fusion_max'] = df[['fg', 'rg']].max(axis=1)
df['fusion_voting'] = (df['fg'] > 0.5).astype(int) + (df['rg'] > 0.5).astype(int)
df['fusion_bayesian_or'] = 1 - (1 - df['fg']) * (1 - df['rg'])
print(f'Valid records: {len(df)}')


In [ ]:
def bootstrap_auroc(y, scores, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    aurocs = []
    for _ in range(n):
        idx = rng.choice(len(y), size=len(y), replace=True)
        ys = y[idx]; ss = scores[idx]
        if len(set(ys)) < 2: continue
        aurocs.append(roc_auc_score(ys, ss))
    arr = np.array(aurocs)
    return float(arr.mean()), float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

def paired_lift_ci(y, single_scores, ensemble_scores, n=1000, seed=42):
    """Paired bootstrap on (ensemble - single) AUROC. Tighter than independent CI."""
    rng = np.random.default_rng(seed)
    lifts = []
    for _ in range(n):
        idx = rng.choice(len(y), size=len(y), replace=True)
        ys = y[idx]
        if len(set(ys)) < 2: continue
        try:
            a_single = roc_auc_score(ys, single_scores[idx])
            a_ens = roc_auc_score(ys, ensemble_scores[idx])
            lifts.append(a_ens - a_single)
        except: continue
    arr = np.array(lifts)
    return float(arr.mean()), float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

methods = ['fg', 'rg', 'fusion_weighted_avg', 'fusion_max', 'fusion_voting', 'fusion_bayesian_or']
ensemble_methods = ['fusion_weighted_avg', 'fusion_max', 'fusion_voting', 'fusion_bayesian_or']

per_source = {}
for src in sorted(df['src'].unique()):
    src_df = df[df['src'] == src]
    if len(src_df) < 20 or len(set(src_df['target'])) < 2:
        print(f'\n=== {src} === SKIP (n={len(src_df)}, target_classes={len(set(src_df["target"]))})')
        continue
    y = src_df['target'].values
    pos_count = int(y.sum())
    if pos_count < 10 or (len(y) - pos_count) < 10:
        print(f'\n=== {src} === SKIP (class imbalance: pos={pos_count}, neg={len(y)-pos_count})')
        continue
    print(f'\n=== {src} (n={len(src_df)}, incorrect_rate={y.mean():.2%}) ===')
    src_results = {}
    for m in methods:
        scores = src_df[m].values.astype(float)
        auroc = roc_auc_score(y, scores)
        boot_mean, lo, hi = bootstrap_auroc(y, scores, n=CFG['bootstrap_n'])
        src_results[m] = {'auroc': auroc, 'ci_lo': lo, 'ci_hi': hi}
        print(f'  {m:<26} {auroc:.4f}   [{lo:.3f}, {hi:.3f}]')
    best_single = max(src_results['fg']['auroc'], src_results['rg']['auroc'])
    best_ens_method = max(ensemble_methods, key=lambda m: src_results[m]['auroc'])
    best_ensemble = src_results[best_ens_method]['auroc']
    lift = best_ensemble - best_single
    single_better = 'fg' if src_results['fg']['auroc'] >= src_results['rg']['auroc'] else 'rg'
    lift_mean, lift_lo, lift_hi = paired_lift_ci(
        y, src_df[single_better].values.astype(float),
        src_df[best_ens_method].values.astype(float),
        n=CFG['bootstrap_n']
    )
    if lift >= CFG['lift_threshold_pp'] and lift_lo > 0:
        verdict_emoji = '🟢'; verdict = 'survives'
    elif lift <= -CFG['lift_threshold_pp'] and lift_hi < 0:
        verdict_emoji = '🔴'; verdict = 'hurts'
    else:
        verdict_emoji = '⚪'; verdict = 'null'
    print(f'  best_single={best_single:.4f} ({single_better})')
    print(f'  best_ensemble={best_ensemble:.4f} ({best_ens_method.replace("fusion_","")})')
    print(f'  lift={lift:+.4f}  paired CI=[{lift_lo:+.4f}, {lift_hi:+.4f}]  {verdict_emoji} {verdict}')
    fg_rg_corr = float(src_df[['fg', 'rg']].corr().iloc[0, 1])
    print(f'  FG-RG correlation: {fg_rg_corr:+.3f}')
    per_source[src] = {
        'n': len(src_df),
        'incorrect_rate': float(y.mean()),
        'methods': src_results,
        'best_single': float(best_single),
        'best_single_method': single_better,
        'best_ensemble': float(best_ensemble),
        'best_ensemble_method': best_ens_method,
        'lift': float(lift),
        'lift_paired_ci_lo': lift_lo,
        'lift_paired_ci_hi': lift_hi,
        'verdict': verdict,
        'verdict_emoji': verdict_emoji,
        'fg_rg_correlation': fg_rg_corr,
    }


In [ ]:
# RG calibration cross-check vs nb32 (StrategyQA known reference)
if 'strategyqa' in per_source:
    rg_stq = per_source['strategyqa']['methods']['rg']['auroc']
    expected = CFG['rg_strategyqa_ref']
    delta = rg_stq - expected
    print(f'RG StrategyQA cross-check:')
    print(f'  measured: {rg_stq:.4f}')
    print(f'  expected (nb32): {expected:.4f}')
    print(f'  delta: {delta:+.4f}')
    if abs(delta) > 0.10:
        print(f'  ⚠️  delta > 0.10 — RG measurement may be unreliable. Different sample? Different probe behavior?')
    else:
        print(f'  ✓ within tolerance')

# Fusion-method consistency: is the best ensemble method the same across datasets?
best_methods = [per_source[s]['best_ensemble_method'] for s in per_source]
from collections import Counter
method_counts = Counter(best_methods)
print(f'\nBest ensemble method per dataset: {dict(zip(per_source.keys(), best_methods))}')
print(f'Method consistency: {method_counts}')
if len(method_counts) == 1:
    print('  ✓ same fusion method dominates all datasets — robust')
elif len(method_counts) == 2:
    print('  🟡 fusion method varies — moderate fusion-method overfit risk')
else:
    print('  ⚠️  every dataset has a different best fusion method — fusion-method overfit suspected')


In [ ]:
# Calibration: ECE per fusion method per dataset
def ece_score(y_true, scores, n_bins=10):
    """Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (scores >= bins[i]) & (scores < bins[i+1] if i < n_bins-1 else scores <= bins[i+1])
        if mask.sum() == 0: continue
        bin_acc = y_true[mask].mean()
        bin_conf = scores[mask].mean()
        ece += (mask.sum() / len(scores)) * abs(bin_acc - bin_conf)
    return float(ece)

calibration_methods = ['fg', 'rg', 'fusion_weighted_avg', 'fusion_bayesian_or']
print(f'{"source":<14} ' + ' '.join(f'{m[:14]:>14}' for m in calibration_methods))
print('-' * 76)
for src in per_source:
    src_df = df[df['src'] == src]
    y = src_df['target'].values
    eces = []
    for m in calibration_methods:
        scores = src_df[m].values.astype(float)
        if scores.max() > 1.01:
            scores = scores / scores.max()
        eces.append(ece_score(y, scores))
    per_source[src]['ece'] = dict(zip(calibration_methods, eces))
    print(f'{src:<14} ' + ' '.join(f'{e:>14.4f}' for e in eces))


In [ ]:
# Aggregate verdict — 5-state
n_survives = sum(1 for r in per_source.values() if r['verdict'] == 'survives')
n_hurts = sum(1 for r in per_source.values() if r['verdict'] == 'hurts')
n_null = sum(1 for r in per_source.values() if r['verdict'] == 'null')
n_total = len(per_source)
lifts = np.array([r['lift'] for r in per_source.values()])
mean_lift = float(lifts.mean()) if len(lifts) > 0 else 0.0

# Bootstrap CI on mean lift across datasets
rng = np.random.default_rng(42)
boot_means = []
for _ in range(CFG['bootstrap_n']):
    idx = rng.choice(len(lifts), size=len(lifts), replace=True)
    boot_means.append(lifts[idx].mean())
mean_lift_lo = float(np.percentile(boot_means, 2.5))
mean_lift_hi = float(np.percentile(boot_means, 97.5))

print(f'\n=== AGGREGATE ===')
print(f'Datasets evaluated: {n_total}')
print(f'  🟢 survives: {n_survives}')
print(f'  ⚪ null:     {n_null}')
print(f'  🔴 hurts:    {n_hurts}')
print(f'Mean lift: {mean_lift:+.4f}  CI=[{mean_lift_lo:+.4f}, {mean_lift_hi:+.4f}]')

if n_hurts >= 1:
    aggregate = '🔴 HURTS — pull ensemble axis from ProbeBench, walk back public claim'
    paper4_status = 'rejected'; paper5_status = 'rejected'
elif n_survives == n_total and mean_lift > 0.03 and mean_lift_lo > 0:
    aggregate = '🟢 STRONG — ensemble generalizes broadly OOD. Paper-4 + paper-5 ship.'
    paper4_status = 'ship'; paper5_status = 'ship'
elif n_survives >= 2 and mean_lift >= 0.02:
    aggregate = '🟢 STANDARD — ensemble generalizes with declared scope. Paper-4 ships.'
    paper4_status = 'ship_with_scope'; paper5_status = 'foundation_ok'
elif n_survives >= 1 and n_hurts == 0:
    aggregate = '🟡 MIXED — partial generalization. Paper-4 weak. ProbePack needs guardrails.'
    paper4_status = 'weak'; paper5_status = 'risky'
else:
    aggregate = '⚪ NULL — ensemble = single in OOD. Walk back claim. Single probes only.'
    paper4_status = 'rejected'; paper5_status = 'foundation_weak'

print(f'\nAggregate verdict: {aggregate}')
print(f'  paper-4 (ProbePack ensemble): {paper4_status}')
print(f'  paper-5 (Probe-Gated Memory): {paper5_status}')


## Phase 7 — Visualization (3 panels: AUROC, lift, calibration)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
srcs = list(per_source.keys())
x = np.arange(len(srcs))
w = 0.27

# Panel 1 — AUROC: FG vs RG vs best ensemble per dataset
fg_aurocs = [per_source[s]['methods']['fg']['auroc'] for s in srcs]
rg_aurocs = [per_source[s]['methods']['rg']['auroc'] for s in srcs]
ens_aurocs = [per_source[s]['best_ensemble'] for s in srcs]
fg_los = [per_source[s]['methods']['fg']['ci_lo'] for s in srcs]
fg_his = [per_source[s]['methods']['fg']['ci_hi'] for s in srcs]
rg_los = [per_source[s]['methods']['rg']['ci_lo'] for s in srcs]
rg_his = [per_source[s]['methods']['rg']['ci_hi'] for s in srcs]
fg_err = [[a-l for a,l in zip(fg_aurocs, fg_los)], [h-a for h,a in zip(fg_his, fg_aurocs)]]
rg_err = [[a-l for a,l in zip(rg_aurocs, rg_los)], [h-a for h,a in zip(rg_his, rg_aurocs)]]
axes[0].bar(x - w, fg_aurocs, w, yerr=fg_err, label='FG alone', color='#3b82f6', alpha=0.85, capsize=3)
axes[0].bar(x, rg_aurocs, w, yerr=rg_err, label='RG alone', color='#f59e0b', alpha=0.85, capsize=3)
axes[0].bar(x + w, ens_aurocs, w, label='best ensemble', color='#10b981', alpha=0.85)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='chance')
axes[0].set_xticks(x); axes[0].set_xticklabels(srcs, rotation=0, fontsize=10)
axes[0].set_ylabel('AUROC (detect incorrect)')
axes[0].set_ylim(0.4, 1.0)
axes[0].set_title('Per-dataset AUROC: FG vs RG vs best ensemble')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Panel 2 — Lift with paired bootstrap CI
lift_means = [per_source[s]['lift'] for s in srcs]
lift_los = [per_source[s]['lift_paired_ci_lo'] for s in srcs]
lift_his = [per_source[s]['lift_paired_ci_hi'] for s in srcs]
errs = [[m-lo for m,lo in zip(lift_means, lift_los)],
        [hi-m for hi,m in zip(lift_his, lift_means)]]
colors_lift = []
for s in srcs:
    v = per_source[s]['verdict']
    colors_lift.append('#10b981' if v=='survives' else ('#ef4444' if v=='hurts' else '#9ca3af'))
axes[1].bar(srcs, lift_means, yerr=errs, color=colors_lift, alpha=0.85, capsize=8)
axes[1].axhline(0.03, color='green', linestyle=':', alpha=0.5, label='+3pp threshold')
axes[1].axhline(0, color='black', alpha=0.5)
axes[1].axhline(-0.03, color='red', linestyle=':', alpha=0.5, label='−3pp threshold')
axes[1].axhline(mean_lift, color='blue', linestyle='--', alpha=0.7, label=f'mean lift={mean_lift:+.3f}')
axes[1].set_ylabel('Ensemble lift over best single (paired bootstrap)')
axes[1].set_title(f'Cross-distribution lift  |  mean CI=[{mean_lift_lo:+.3f}, {mean_lift_hi:+.3f}]')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Panel 3 — ECE per method per dataset
calib_methods_short = ['fg', 'rg', 'wavg', 'bay_or']
calib_methods_full = ['fg', 'rg', 'fusion_weighted_avg', 'fusion_bayesian_or']
ece_data = np.array([[per_source[s]['ece'][m] for m in calib_methods_full] for s in srcs])
for i, s in enumerate(srcs):
    axes[2].plot(calib_methods_short, ece_data[i], marker='o', linewidth=2, markersize=8, label=s)
axes[2].axhline(0.10, color='red', linestyle=':', alpha=0.5, label='ECE=0.10 (poor)')
axes[2].axhline(0.05, color='orange', linestyle=':', alpha=0.5, label='ECE=0.05 (ok)')
axes[2].set_ylabel('Expected Calibration Error (lower = better)')
axes[2].set_title('Calibration per method per dataset')
axes[2].set_ylim(0, max(0.20, ece_data.max() * 1.1))
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'fig_cross_distribution_main.png', dpi=150, bbox_inches='tight')
plt.show()


## Phase 8 — FINAL VERDICT + HF push


In [ ]:
verdict_obj = {
    'experiment': 'nb46 cross-distribution multi-probe ensemble',
    'hypothesis': 'nb45 ensemble +6.7pp generalizes to OOD datasets',
    'datasets': sorted(per_source.keys()),
    'n_per_dataset_target': CFG['n_per_dataset'],
    'n_total_valid': len(df),
    'lift_threshold_pp': CFG['lift_threshold_pp'],
    'per_source': per_source,
    'aggregate': {
        'n_survives': n_survives,
        'n_null': n_null,
        'n_hurts': n_hurts,
        'mean_lift': mean_lift,
        'mean_lift_ci_lo': mean_lift_lo,
        'mean_lift_ci_hi': mean_lift_hi,
        'verdict': aggregate,
        'paper4_status': paper4_status,
        'paper5_status': paper5_status,
    },
    'fg_rg_correlation_overall': float(df[['fg', 'rg']].corr().iloc[0, 1]),
    'diagnostics': {
        'has_think_per_source': {s: float(d['thinking']/max(1,d['total'])) for s,d in diag.items()},
        'rg_strategyqa_measured_vs_ref': {
            'measured': per_source.get('strategyqa', {}).get('methods', {}).get('rg', {}).get('auroc'),
            'reference': CFG['rg_strategyqa_ref'],
        },
    },
}
(OUT / 'FINAL_VERDICT.json').write_text(json.dumps(verdict_obj, indent=2, default=str))
print(json.dumps(verdict_obj, indent=2, default=str))


In [ ]:
api = HfApi()
try: create_repo(CFG['output_repo'], repo_type='dataset', private=False, exist_ok=True, token=HF_TOKEN)
except Exception as e: print(e)

readme_lines = [
    '---',
    'license: apache-2.0',
    'tags:',
    '- inference-ensemble',
    '- multi-probe',
    '- qwen36-27b',
    '- cross-distribution',
    '- ood',
    '- probebench',
    '---',
    '',
    '# nb46 — Cross-Distribution Multi-Probe Ensemble',
    '',
    f'**Verdict**: {aggregate}',
    '',
    f'- paper-4 (ProbePack ensemble): {paper4_status}',
    f'- paper-5 (Probe-Gated Memory): {paper5_status}',
    f'- Mean lift: {mean_lift:+.4f}  CI=[{mean_lift_lo:+.4f}, {mean_lift_hi:+.4f}]',
    '',
    '## Setup',
    '',
    '- Base Qwen3.6-27B (no LoRA, inference middleware test)',
    f'- {len(df)} valid generations across 3 OOD datasets (TruthfulQA gen, StrategyQA, TriviaQA rc.nocontext)',
    '- FG L31 + RG L55 probes (frozen, from nb45)',
    '- 4 fusion methods: weighted_avg, max, voting, bayesian_or',
    '- Claude Haiku judge for ground truth',
    '',
    '## Per-dataset',
    '',
    '| Dataset | n | FG AUROC | RG AUROC | Best ensemble | Lift | CI | Verdict |',
    '|---|---|---|---|---|---|---|---|',
]
for s, r in per_source.items():
    readme_lines.append(
        f'| {s} | {r["n"]} | {r["methods"]["fg"]["auroc"]:.3f} | '
        f'{r["methods"]["rg"]["auroc"]:.3f} | {r["best_ensemble"]:.3f} ({r["best_ensemble_method"].replace("fusion_","")}) | '
        f'{r["lift"]:+.3f} | [{r["lift_paired_ci_lo"]:+.3f}, {r["lift_paired_ci_hi"]:+.3f}] | {r["verdict_emoji"]} {r["verdict"]} |'
    )
readme_lines += [
    '',
    '## Aggregate verdict states',
    '',
    '| State | Criterion |',
    '|---|---|',
    '| 🟢 STRONG | 3/3 survive, mean lift > +3pp, CI excludes 0 |',
    '| 🟢 STANDARD | 2/3 survive, mean lift ≥ +2pp |',
    '| 🟡 MIXED | 1/3 survive, no hurts |',
    '| ⚪ NULL | 0/3 survive, mean ≈ 0 |',
    '| 🔴 HURTS | ≥1 hurts |',
    '',
    '## Files',
    '',
    '- `FINAL_VERDICT.json` — structured verdict with all per-dataset numbers',
    '- `fig_cross_distribution_main.png` — 3-panel: AUROC, lift, calibration',
    '- `scored.jsonl` — per-prompt records (probe scores, judge labels)',
    '',
    'See sibling dataset `caiovicentino1/openinterp-45-inference-ensemble` for in-distribution baseline.',
]
(OUT / 'README.md').write_text('\n'.join(readme_lines))

try:
    api.upload_folder(folder_path=str(OUT), repo_id=CFG['output_repo'],
                      repo_type='dataset', token=HF_TOKEN,
                      commit_message=f'nb46 cross-distribution: {aggregate}',
                      allow_patterns=['README.md', 'FINAL_VERDICT.json',
                                      'fig_*.png', 'scored.jsonl'])
    print('✓ pushed')
except Exception as e:
    print(f'HF push failed: {e}')


## Done — interpretation guide

**🟢 STRONG**: ProbePack OOD claim survives. Update `/products/probepack` with cross-dataset evidence. Begin paper-5 sketch (Probe-Gated Memory). Tweet result + cite ReasoningBank as adjacent work.

**🟢 STANDARD**: ProbePack works in declared scope (whichever 2/3 survived). Update product page with explicit scope statement. Paper-4 viable as workshop submission.

**🟡 MIXED**: Walk back universal-middleware framing. ProbePack becomes 'works for X-style tasks, untested elsewhere'. Paper-4 becomes 'fusion is domain-bound' negative-ish result. Still publishable as honest finding.

**⚪ NULL**: nb45 was within-distribution effect. Walk back ensemble claim publicly. Single probes remain the product. Paper-4 dies but honest-negative one-pager possible: 'ensemble lift does not generalize OOD on Qwen3.6-27B with FG/RG probes'.

**🔴 HURTS**: Pull ensemble axis from ProbeBench v0.0.2. Public correction post. Paper-4 dies. This is the worst outcome but knowing it now beats finding it via a paying customer.

Whatever the verdict, push results to HF (already done by Phase 8). Honest negatives strengthen the trajectory more than fake wins.
